# MirrorTopology Step 1 — Phase B・B-3-2 A：登録 12 位置 asset と新 9 点の circle 幾何受入れ（v0.2）
E2（正式 1e-4 格子・streamed two-pass：**各 size の生成 9 step＋intake の再生成 9 step，3 size で selector 6 回**）／E7／E8 の 12 位置 manifest asset を source-bound registry から生成・再生成検証し，凍結 A7 手順（fresh・clean・pinned CMBtopology）で全 12 点 × surviving size の circle 幾何を判定する。重い処理の前に外部 checkout を検査する。所要時間の目安：E2 で 2.5〜5 h（Colab CPU；実測はこの実行が初）。label なし・共分散なし（A11 の共分散変換・clone 同値の検査は含まない）。

In [ ]:
# --- 0. OUTER LAUNCHER LOCK (the only editable cell)
REPO_URL = 'https://github.com/tsujikeita/mirror-topology.git'
REPO_COMMIT = '<full 40-hex commit of the verification target>'
EXPECTED_INVENTORY_SHA256 = '<sha256 of engine/phaseB/B2_completion_inventory.json inside that commit>'
LAUNCHER_ID = 'MirrorTopology_Step1_B3_2A_assets_v0.1'


In [ ]:
# --- 1. fresh scratch checkout; clean tree; inventory bound; pins / script bound through the inventory
import subprocess, sys, os, json, hashlib, shutil, time, re
sha=lambda p: hashlib.sha256(open(p,'rb').read()).hexdigest()
assert re.fullmatch(r'[0-9a-f]{40}', REPO_COMMIT) and re.fullmatch(r'[0-9a-f]{64}', EXPECTED_INVENTORY_SHA256), 'launcher lock not filled'
RUN=f'/content/b3_2A_runs/{time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())}'; SCRATCH=f'{RUN}/scratch'; OUT=f'{RUN}/out'; os.makedirs(OUT)
subprocess.run(['git','clone','-q',REPO_URL,SCRATCH],check=True); subprocess.run(['git','-C',SCRATCH,'checkout','-q',REPO_COMMIT],check=True)
head=subprocess.check_output(['git','-C',SCRATCH,'rev-parse','HEAD']).decode().strip(); assert head==REPO_COMMIT, head
assert subprocess.check_output(['git','-C',SCRATCH,'status','--porcelain']).decode().strip()=='', 'scratch tree not clean'
MT=SCRATCH; PHASEB=f'{MT}/engine/phaseB'; INV=f'{PHASEB}/B2_completion_inventory.json'; inv_sha=sha(INV); assert inv_sha==EXPECTED_INVENTORY_SHA256, inv_sha
inv=json.load(open(INV)); PINS=f'{PHASEB}/b3/b3_2_pins.json'; SCRIPT=f'{PHASEB}/b3/b3_2_assets.py'
assert sha(PINS)==inv['b3_sha256']['b3/b3_2_pins.json'] and sha(SCRIPT)==inv['b3_sha256'][f'b3/{os.path.basename(SCRIPT)}']
pins=json.load(open(PINS)); assert 'repo' not in pins and pins['engine_version']==inv['engine_version'] and pins['schema']=='b3_2_pins_v1'
lock=dict(launcher_id=LAUNCHER_ID, repo_url=REPO_URL, commit=head, inventory_sha256=inv_sha, pins_sha256=sha(PINS), engine_version=inv['engine_version'], run_dir=RUN); json.dump(lock, open(f'{OUT}/launcher_lock.json','w'), indent=1); print(lock)


In [ ]:
# --- 2. environment lock
ex=pins['environment']
subprocess.run(['pip','install','-q',f"numpy=={ex['numpy']}",f"scipy=={ex['scipy']}",f"healpy=={ex['healpy']}",f"pot=={ex['pot']}",f"camb=={ex['camb']}",'threadpoolctl'],check=True)
os.environ['OPENBLAS_NUM_THREADS']='2'; os.environ['OMP_NUM_THREADS']='2'
import platform, numpy, scipy, healpy, ot, camb
live=dict(python=platform.python_version(), numpy=numpy.__version__, scipy=scipy.__version__, healpy=healpy.__version__, pot=ot.__version__, camb=camb.__version__)
mism={k:(live[k],ex[k]) for k in live if live[k]!=ex[k]}; assert not mism, f'environment lock failed: {mism}'; print('environment lock OK', live)


In [ ]:
# --- 3. pinned CMBtopology: FRESH isolated checkout; origin / commit / clean tree verified BEFORE the heavy run
CT=f'{RUN}/CMBtopology_pinned'
subprocess.run(['git','clone','-q',pins['a7_script']['cmbtopology_url'],CT],check=True); subprocess.run(['git','-C',CT,'checkout','-q',pins['a7_script']['cmbtopology_commit']],check=True)
ct_head=subprocess.check_output(['git','-C',CT,'rev-parse','HEAD']).decode().strip(); ct_origin=subprocess.check_output(['git','-C',CT,'remote','get-url','origin']).decode().strip().rstrip('/').removesuffix('.git'); ct_clean=subprocess.check_output(['git','-C',CT,'status','--porcelain']).decode().strip()==''
assert ct_head==pins['a7_script']['cmbtopology_commit'] and ct_origin==pins['a7_script']['cmbtopology_url'].rstrip('/').removesuffix('.git') and ct_clean, (ct_head, ct_origin, ct_clean)
assert sha(f"{MT}/{pins['a7_script']['path']}")==pins['a7_script']['sha256'], 'frozen A7 script SHA differs'
json.dump(dict(cmbtopology_head=ct_head, origin=ct_origin, clean=ct_clean, a7_script_sha256=pins['a7_script']['sha256']), open(f'{OUT}/external_checkout_lock.json','w'), indent=1); print('CMBtopology pinned OK', ct_head)


In [ ]:
# --- 4. assets + geometry (long: E2 formal lattice)
AO=f'{OUT}/assets'
rc=subprocess.run([sys.executable,SCRIPT,'--mt',MT,'--phaseb',PHASEB,'--out',AO,'--ct',CT,'--profile','assets_official'],capture_output=True,text=True)
open(f'{OUT}/launcher_script_stdout.txt','w').write(rc.stdout); open(f'{OUT}/launcher_script_stderr.txt','w').write(rc.stderr); print(rc.stdout[-2500:])
rm=json.load(open(f'{AO}/b3_2_assets_run_manifest.json')); script_ok=bool(rc.returncode==0 and rm.get('B3_2A_PASS') is True); print('script rc', rc.returncode, 'B3_2A_PASS', rm.get('B3_2A_PASS'), rm.get('failures'))


In [ ]:
# --- final record (composed; excludes itself) + zip for the audit
def inventory(root, exclude=()):
    out={}
    for d,_,fs in os.walk(root):
        for f in fs:
            p=os.path.join(d,f); rel=os.path.relpath(p, root)
            if rel in exclude or '/ckpt/' in rel or rel.startswith('ckpt/'): continue
            out[rel]=dict(sha256=sha(p), bytes=os.path.getsize(p))
    return out
final=dict(launcher=lock, B3_2A_PASS=bool(script_ok), stages=dict(checkout=True, environment=live, script_returncode=rc.returncode, script_pass=rm.get('B3_2A_PASS'), script_gates=rm.get('gates'), script_failures=rm.get('failures'), timings=rm.get('timings')))
final['output_inventory']=inventory(OUT, exclude=('b3_2A_final_record.json','b3_2A_return_list.json'))
json.dump(final, open(f'{OUT}/b3_2A_final_record.json','w'), indent=1); json.dump(dict(final_record_sha256=sha(f'{OUT}/b3_2A_final_record.json'), files=final['output_inventory']), open(f'{OUT}/b3_2A_return_list.json','w'), indent=1)
print(json.dumps({k:final[k] for k in ('B3_2A_PASS',)}, indent=1), 'run dir:', RUN)
import shutil
from google.colab import files
p = shutil.make_archive(f'/content/b3_2A_out_{REPO_COMMIT[:12]}', 'zip', root_dir=OUT); print(p, os.path.getsize(p)); files.download(p)
